# 01 - Explorar os dados brutos

In [0]:
from pyspark.sql import functions as F

In [0]:
RAW = "/Volumes/workspace/baywheels/raw"
DIA = "2026-11-07"

COLETA_LIGADA_EM = "2026-10-31T21:35:00"

## 1. Coletas por dia x relatórios de cobertura

In [0]:
status_todos = spark.read.json(f"{RAW}/station_status/")

por_dia = status_todos.groupBy(F.col("date").cast("string").alias("data")).agg(
    F.count("*").alias("linhas_no_volume"),
    # O relatório não conta o Test manual de 31/10; aqui também não
    F.sum(F.when(F.col("coletado_em") >= COLETA_LIGADA_EM, 1).otherwise(0)).alias("coletas_do_scheduler"),
)

# O relatório é JSON indentado (várias linhas por arquivo): multiLine
relatorios = spark.read.option("multiLine", True).json(f"{RAW}/relatorios/cobertura/").select(
    "data", "esperadas", "encontradas", "cobertura", "sem_comunicacao_1h"
)

comparacao = por_dia.join(relatorios, "data", "full").orderBy("data")
display(comparacao)

data,linhas_no_volume,coletas_do_scheduler,esperadas,encontradas,cobertura,sem_comunicacao_1h
2026-10-31,30,29,29,29,1.0,35
2026-11-01,288,288,288,288,1.0,33
2026-11-02,288,288,288,288,1.0,34
2026-11-03,288,288,288,288,1.0,27
2026-11-04,288,288,288,288,1.0,29
2026-11-05,288,288,288,288,1.0,27
2026-11-06,288,288,288,288,1.0,29
2026-11-07,288,288,288,288,1.0,28
2026-11-08,288,288,288,288,1.0,28


In [0]:
divergentes = comparacao.filter(~F.col("coletas_do_scheduler").eqNullSafe(F.col("encontradas")))
assert divergentes.count() == 0, "Contagem no Volume diferente da do relatório"
print("Contagem por dia igual à dos relatórios de cobertura")

Contagem por dia igual à dos relatórios de cobertura


## 2. Uma linha por estação por coleta

In [0]:
status = spark.read.json(f"{RAW}/station_status/date={DIA}/")
status.printSchema()

root
 |-- coletado_em: string (nullable = true)
 |-- feed: string (nullable = true)
 |-- resposta: struct (nullable = true)
 |    |-- data: struct (nullable = true)
 |    |    |-- stations: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- is_installed: long (nullable = true)
 |    |    |    |    |-- is_renting: long (nullable = true)
 |    |    |    |    |-- is_returning: long (nullable = true)
 |    |    |    |    |-- last_reported: long (nullable = true)
 |    |    |    |    |-- num_bikes_available: long (nullable = true)
 |    |    |    |    |-- num_bikes_disabled: long (nullable = true)
 |    |    |    |    |-- num_docks_available: long (nullable = true)
 |    |    |    |    |-- num_docks_disabled: long (nullable = true)
 |    |    |    |    |-- num_ebikes_available: long (nullable = true)
 |    |    |    |    |-- num_scooters_available: long (nullable = true)
 |    |    |    |    |-- num_scooters_unavailable: long (nullable

In [0]:
estacoes = (
    status.select(
        F.to_timestamp("coletado_em").alias("coletado_em"),
        F.col("resposta.last_updated").alias("feed_atualizado"),
        F.explode("resposta.data.stations").alias("estacao"),
    )
    .select("coletado_em", "feed_atualizado", "estacao.*")
    # Idade do último sinal, no relógio do próprio feed (igual ao relatório)
    .withColumn("idade_min", (F.col("feed_atualizado") - F.col("last_reported")) / 60)
)

por_coleta = estacoes.groupBy("coletado_em").count()
display(por_coleta.agg(F.count("*").alias("coletas"), F.min("count").alias("min_estacoes"), F.max("count").alias("max_estacoes")))


coletas,min_estacoes,max_estacoes
288,654,655


## 3. Quanto se repete entre coletas

In [0]:
total = estacoes.count()
distintos = estacoes.select("station_id", "last_reported").distinct().count()
print(f"{total:,} linhas, {distintos:,} relatos distintos → {1 - distintos / total:.1%} repetido")

188,615 linhas, 179,410 relatos distintos → 4.9% repetido


In [0]:
# Quantos relatos novos cada estação manda por dia (288 = mudou em toda coleta)
relatos = estacoes.groupBy("station_id").agg(F.countDistinct("last_reported").alias("relatos_no_dia"))
display(
    relatos.select(
        F.percentile_approx("relatos_no_dia", [0.1, 0.25, 0.5, 0.75, 0.9]).alias("p10_p25_p50_p75_p90"),
        F.sum(F.when(F.col("relatos_no_dia") == 1, 1).otherwise(0)).alias("estacoes_com_1_relato"),
    )
)

p10_p25_p50_p75_p90,estacoes_com_1_relato
"List(288, 288, 288, 288, 288)",28


In [0]:
# Conferência: mesmo last_reported implica mesmo estoque? Se não, o last_reported não é confiável como chave
anomalias = (
    estacoes.groupBy("station_id", "last_reported")
    .agg(F.countDistinct("num_bikes_available", "num_docks_available").alias("estados"))
    .filter("estados > 1")
)
print(f"{anomalias.count()} relatos com mais de um estoque para o mesmo last_reported")


53 relatos com mais de um estoque para o mesmo last_reported


## 4. Quais são as estações mudas?

In [0]:
def ultimo_retrato(feed, campo):
    # coletado_em é ISO 8601 em UTC: ordem de texto = ordem de tempo
    bruto = spark.read.json(f"{RAW}/{feed}/date={DIA}/")
    ultimo = bruto.orderBy(F.desc("coletado_em")).limit(1)
    return ultimo.select(F.explode(f"resposta.data.{campo}").alias("item")).select("item.*")


cadastro = ultimo_retrato("station_information", "stations").select(
    "station_id", "short_name", "name", "capacity", "region_id"
)
regioes = ultimo_retrato("system_regions", "regions").select(
    "region_id", F.col("name").alias("regiao")
)

ultima_coleta = estacoes.agg(F.max("coletado_em")).first()[0]
retrato = (
    estacoes.filter(F.col("coletado_em") == ultima_coleta)
    .join(cadastro, "station_id", "left")
    .join(regioes, "region_id", "left")
)

print(f"{retrato.filter(F.col('name').isNull()).count()} estações do status sem cadastro no information")


1 estações do status sem cadastro no information


In [0]:
mudas = retrato.filter("idade_min > 60")
display(
    mudas.select(
        "station_id",
        "short_name",
        "name",
        "regiao",
        "capacity",
        "is_installed",
        "is_renting",
        "num_bikes_available",
        F.round(F.col("idade_min") / 60 / 24, 1).alias("dias_sem_sinal"),
    ).orderBy(F.desc("dias_sem_sinal"))
)

station_id,short_name,name,regiao,capacity,is_installed,is_renting,num_bikes_available,dias_sem_sinal
2194949114935930778,null,VIRTUAL STATION: Bayshore Caltrain,null,6,1,1,0,47.2
1841972814951361206,null,VIRTUAL STATION: South Lake,null,10,1,1,0,47.2
2194949114935930776,null,VIRTUAL STATION: Howard St at Beale St,null,10,1,1,1,47.2
2207126171341300786,null,VIRTUAL STATION: Pier 27,null,20,1,1,0,47.2
1875048568201626296,null,VIRTUAL STATION: Taraval St at 27th Ave to 25th Ave,null,5,1,1,0,47.2
1875048568201626286,null,VIRTUAL STATION: Sloat Blvd at The Great Highway to 46th Ave,null,6,1,1,0,47.2
1880942617512231608,null,VIRTUAL STATION: Lloyd Lake,null,16,1,1,9,47.2
1875048572496593608,null,VIRTUAL STATION: Ocean Avenue East: to Plymouth Ave,null,5,1,1,2,47.2
1880193329986742718,null,VIRTUAL STATION: Taraval St: 41st St to 40th St,null,4,1,1,1,47.2
1875395906711399396,null,VIRTUAL STATION: Forest Hill Muni Station,null,4,1,1,2,47.2


In [0]:
display(mudas.groupBy("regiao").count().orderBy(F.desc("count")))

regiao,count
null,28


In [0]:
# Tem que bater com o relatório do mesmo dia
relatorio_do_dia = relatorios.filter(F.col("data") == DIA).first()
print(f"Notebook: {mudas.count()} mudas · relatório de {DIA}: {relatorio_do_dia['sem_comunicacao_1h']}")


Notebook: 28 mudas · relatório de 2026-11-07: 28


In [0]:
virtuais = cadastro.withColumn("virtual", F.col("name").startswith("VIRTUAL STATION")).select("station_id", "virtual")

# As 53 anomalias (mesmo last_reported, estoque diferente) vêm das virtuais?
display(anomalias.join(virtuais, "station_id", "left").groupBy("virtual").count())

virtual,count
true,18
false,35


In [0]:
# Sem comunicação por dia, separando virtuais das físicas (o relatório ainda conta as duas juntas)
ultima_por_dia = (
    status_todos.withColumn("dia", F.col("date").cast("string"))
    .groupBy("dia")
    .agg(F.max("coletado_em").alias("coletado_em"))
    .join(status_todos.withColumn("dia", F.col("date").cast("string")), ["dia", "coletado_em"])
    .select("dia", F.col("resposta.last_updated").alias("feed_atualizado"), F.explode("resposta.data.stations").alias("e"))
    .select("dia", "feed_atualizado", "e.station_id", "e.last_reported")
    .filter((F.col("feed_atualizado") - F.col("last_reported")) > 3600)
    .join(virtuais, "station_id", "left")
)
display(
    ultima_por_dia.groupBy("dia")
    .agg(
        F.sum(F.when(F.col("virtual"), 1).otherwise(0)).alias("virtuais"),
        F.sum(F.when(~F.col("virtual") | F.col("virtual").isNull(), 1).otherwise(0)).alias("fisicas_mudas"),
    )
    .orderBy("dia")
)

dia,virtuais,fisicas_mudas
2026-10-31,27,8
2026-11-01,27,6
2026-11-02,27,7
2026-11-03,27,0
2026-11-04,27,2
2026-11-05,27,0
2026-11-06,28,1
2026-11-07,28,0
2026-11-08,28,0


## 5. GBFS × viagens: o `short_name` casa com o `start_station_id`?

In [0]:
viagens = spark.read.csv(f"{RAW}/trips/202609.csv.gz", header=True)
partidas = viagens.groupBy("start_station_id").count()

casamento = partidas.join(
    cadastro.select(F.col("short_name").alias("start_station_id"), "name"), "start_station_id", "left"
)
display(
    casamento.agg(
        F.sum("count").alias("viagens"),
        F.sum(F.when(F.col("start_station_id").isNull(), F.col("count"))).alias("sem_estacao_de_partida"),
        F.sum(F.when(F.col("name").isNotNull(), F.col("count"))).alias("casam_com_o_gbfs"),
    )
)

viagens,sem_estacao_de_partida,casam_com_o_gbfs
607780,88593,514377
